In [1]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
print(df)

     mean radius  mean texture  ...  worst fractal dimension  target
0          17.99         10.38  ...                  0.11890       0
1          20.57         17.77  ...                  0.08902       0
2          19.69         21.25  ...                  0.08758       0
3          11.42         20.38  ...                  0.17300       0
4          20.29         14.34  ...                  0.07678       0
..           ...           ...  ...                      ...     ...
564        21.56         22.39  ...                  0.07115       0
565        20.13         28.25  ...                  0.06637       0
566        16.60         28.08  ...                  0.07820       0
567        20.60         29.33  ...                  0.12400       0
568         7.76         24.54  ...                  0.07039       1

[569 rows x 31 columns]


Explore the dataset with head(), shape, missing values, and the target distribution.

In [2]:
df.head()

   mean radius  mean texture  ...  worst fractal dimension  target
0        17.99         10.38  ...                  0.11890       0
1        20.57         17.77  ...                  0.08902       0
2        19.69         21.25  ...                  0.08758       0
3        11.42         20.38  ...                  0.17300       0
4        20.29         14.34  ...                  0.07678       0

[5 rows x 31 columns]

In [3]:
df.shape

(569, 31)

In [4]:
df.isnull().sum()

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64

In [5]:
df["target"].value_counts()

target
1    357
0    212
Name: count, dtype: int64

Create X and y

In [6]:
X = df.drop(columns="target")
y = df["target"]

Split the data into 80% training / 20% testing using random_state=42 and stratify=y

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Create a pipeline containing:
- StandardScaler
- LogisticRegression(max_iter=5000)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

scaler = StandardScaler()
log_reg = LogisticRegression(max_iter=5000)

pipeline = Pipeline([
    ('scaler', scaler),
    ('log_reg', log_reg)
])



First train the pipeline using Logistic Regression's default C value

In [9]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('log_reg', LogisticRegression(max_iter=5000))])

Record its test accuracy, precision, recall, and F1 score

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, pipeline.predict(X_test))
precision = precision_score(y_test, pipeline.predict(X_test))
recall = recall_score(y_test, pipeline.predict(X_test))
f1 = f1_score(y_test, pipeline.predict(X_test))

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")      
print(f"F1 Score: {f1:.2f}")

Accuracy: 0.98
Precision: 0.99
Recall: 0.99
F1 Score: 0.99


Now use GridSearchCV with 5-fold cross-validation to test

In [11]:
from sklearn.model_selection import GridSearchCV
C_values = [0.01, 0.1, 1, 10, 100]

param_grid = {"log_reg__C": C_values} # __ conects the pipeline step name with the parameter name

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('log_reg',
                                        LogisticRegression(max_iter=5000))]),
             param_grid={'log_reg__C': [0.01, 0.1, 1, 10, 100]},
             scoring='accuracy')

Find the best C.

In [12]:
print("Best C:", grid_search.best_params_["log_reg__C"])
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Best C: 0.1
Best Cross-Validation Accuracy: 0.9802197802197803


Evaluate the best model on the untouched test set using the same four metrics

In [13]:
best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_predictions)
tuned_precision = precision_score(y_test, tuned_predictions)
tuned_recall = recall_score(y_test, tuned_predictions)
tuned_f1 = f1_score(y_test, tuned_predictions)

print("Tuned Model")
print(f"Accuracy: {tuned_accuracy:.2f}")
print(f"Precision: {tuned_precision:.2f}")
print(f"Recall: {tuned_recall:.2f}")
print(f"F1 Score: {tuned_f1:.2f}")

Tuned Model
Accuracy: 0.97
Precision: 0.97
Recall: 0.99
F1 Score: 0.98


Compare the tuned model against your original model

GridSearchCV selected C = 0.1. The tuned model had slightly lower test accuracy than the default model, showing that the best average cross-validation setting does not always produce a higher score on one test split.

In [14]:
print("Default Model")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

print(f"\nBest C: {grid_search.best_params_['log_reg__C']}")

print("\nTuned Model")
print(f"Accuracy: {tuned_accuracy:.2f}")
print(f"Precision: {tuned_precision:.2f}")
print(f"Recall: {tuned_recall:.2f}")
print(f"F1 Score: {tuned_f1:.2f}")

Default Model
Accuracy: 0.98
Precision: 0.99
Recall: 0.99
F1 Score: 0.99

Best C: 0.1

Tuned Model
Accuracy: 0.97
Precision: 0.97
Recall: 0.99
F1 Score: 0.98
